# Caso 1 — Hacking Forums: identidad y tiempo

**Dataset**: RaidForums (2020), BreachForums (2022, 2023), OGUsers (2019, 2020, 2021, 2022)

**Narrativa**: OGUsers era la comunidad más activa de robo y venta de handles de redes sociales ('OG usernames'). Fue brecheada cuatro veces entre 2019 y 2022. Esto nos da algo único: una serie temporal de la misma comunidad, lo que permite ver cómo evoluciona tras cada exposición — qué usuarios desaparecen, quiénes migran, cómo se reconstituye la red.

**Conclusión esperada**: demostrar que una identidad underground no muere con una brecha — migra.

## Estructura
1. [Introducción y contexto](#1-introduccion)
2. [Carga de datos](#2-carga)
3. [Análisis exploratorio](#3-eda)
4. [Análisis de red social](#4-red)
5. [Análisis temporal](#5-temporal)
6. [Estilometría e identidades cruzadas](#6-estilometria)
7. [Embeddings y clustering](#7-embeddings)
8. [Conclusiones](#8-conclusiones)

<a id='1-introduccion'></a>
## 1. Introducción y contexto

### El ecosistema de hacking forums

A diferencia de los foros de carding (orientados a datos de tarjetas), los hacking forums como **RaidForums** y **BreachForums** eran mercados de redistribución de breaches: operadores que vendían accesos a bases de datos robadas, dumps de credenciales y exploits. OGUsers, por su parte, era un nicho específico: la comunidad de robo de handles 'originales' en Instagram, Twitter y Snapchat — names cortos y valiosos que se vendían por miles de dólares.

### Por qué OGUsers es un dataset excepcional

Los cuatro snapshots de OGUsers (2019, 2020, 2021, 2022) nos permiten estudiar **la dinámica de una comunidad a lo largo del tiempo**. Cada brecha es un evento: algunos usuarios desaparecen, otros cambian de handle, la mayoría continúa. La combinación con RaidForums y BreachForums permite rastrear migraciones cross-foro.

### Preguntas de investigación

1. ¿Cómo cambia la estructura de OGUsers tras cada brecha?
2. ¿Cuántos usuarios de OGUsers reaparecen en RaidForums o BreachForums bajo el mismo o distinto handle?
3. ¿Puede la estilometría confirmar identidades cuando el username cambia?

In [ ]:
# Sección 0: imports y configuración
import sys
from pathlib import Path

# Asegurar que la raíz del proyecto esté en sys.path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from difflib import SequenceMatcher

from src.utils import load_forum, list_forums, merge_tables, load_or_compute, RESULTS_DIR, DATA_DIR
from src.stylometry import compare_users, extract_features
from src.timezone import build_user_timezone_profile

# Estilo
plt.style.use('dark_background')
sns.set_palette('muted')

# Directorios de resultados
HF_RESULTS = RESULTS_DIR / 'hacking_forums'
HF_RESULTS.mkdir(parents=True, exist_ok=True)

print(f"Datos:      {DATA_DIR}")
print(f"Resultados: {HF_RESULTS}")

<a id='2-carga'></a>
## 2. Carga de datos

Cargamos los cuatro snapshots de OGUsers y los dos foros adicionales (RaidForums, BreachForums).

**Contrato de normalización para OGUsers (`normalize_snapshots`)**:
- `snapshot_year`: extraído del stem del archivo (e.g. `OGUsers_2019` → 2019); fallback al año mínimo de `joindate`
- `handle_norm`: `username` en lowercase y sin espacios
- `joindate_epoch`: `joindate` normalizado a UTC int epoch
- Para joins cross-snapshot: intersección de columnas comunes entre los 4 dumps

In [ ]:
# Detectar archivos de Hacking Forums
hf_paths = list_forums('Hacking Forums')
print(f"Archivos encontrados: {len(hf_paths)}")
for p in hf_paths:
    print(f"  {p.name}")

In [ ]:
# Cargar todos los foros
raw_forums = {}
for path in hf_paths:
    try:
        dfs = load_forum(path)
        raw_forums[path.stem] = dfs
        u = len(dfs.get('user', []))
        p = len(dfs.get('post', []))
        print(f"  ✓ {path.stem}: {u:,} usuarios, {p:,} posts")
    except Exception as e:
        print(f"  ✗ {path.stem}: {e}")

print(f"\nForos cargados: {len(raw_forums)}")

In [ ]:
def extract_snapshot_year(forum_name: str, user_df: pd.DataFrame) -> int:
    """
    Extrae el año del snapshot del nombre del archivo.
    Fallback: año mínimo de joindate si el stem no contiene un año de 4 dígitos.
    """
    # Buscar patrón de 4 dígitos de año en el nombre
    match = re.search(r'(20\d{2})', forum_name)
    if match:
        return int(match.group(1))
    
    # Fallback: año mínimo de joindate
    if 'joindate' in user_df.columns:
        joindate_numeric = pd.to_numeric(user_df['joindate'], errors='coerce')
        valid = joindate_numeric.dropna()
        if len(valid) > 0:
            min_ts = valid.min()
            if min_ts > 0:
                return pd.Timestamp(min_ts, unit='s').year
    return 0


def normalize_snapshots(forum_dfs: dict) -> dict:
    """
    Normaliza los DataFrames de usuarios para análisis cross-snapshot.
    
    Contrato:
    - handle_norm: username lowercase+strip
    - snapshot_year: año del snapshot (del stem del archivo o del joindate mínimo)
    - joindate_epoch: joindate normalizado a UTC int epoch
    """
    normalized = {}
    for forum_name, dfs in forum_dfs.items():
        if 'user' not in dfs:
            continue
        df = dfs['user'].copy()
        
        # handle_norm
        if 'username' in df.columns:
            df['handle_norm'] = df['username'].astype(str).str.lower().str.strip()
        
        # snapshot_year
        df['snapshot_year'] = extract_snapshot_year(forum_name, df)
        
        # joindate_epoch
        if 'joindate' in df.columns:
            df['joindate_epoch'] = pd.to_numeric(df['joindate'], errors='coerce').fillna(0).astype(int)
        
        normalized[forum_name] = df
    
    return normalized


normalized_users = normalize_snapshots(raw_forums)
print("Normalización completada:")
for name, df in normalized_users.items():
    yr = df['snapshot_year'].iloc[0] if len(df) > 0 else '?'
    print(f"  {name}: {len(df):,} usuarios, año={yr}")

<a id='3-eda'></a>
## 3. Análisis exploratorio

Estadística descriptiva comparada entre los cuatro snapshots de OGUsers y los foros de hacking.

In [ ]:
# Separar OGUsers de los otros foros
ogusers_snapshots = {k: v for k, v in normalized_users.items() if 'ogusers' in k.lower() or 'og' in k.lower()}
other_forums = {k: v for k, v in normalized_users.items() if k not in ogusers_snapshots}

print(f"OGUsers snapshots: {list(ogusers_snapshots.keys())}")
print(f"Otros foros:       {list(other_forums.keys())}")

In [ ]:
# Estadísticas comparadas por foro
all_users_combined = []
for name, df in normalized_users.items():
    df_copy = df.copy()
    df_copy['source_forum'] = name
    all_users_combined.append(df_copy)

if all_users_combined:
    all_users_df = pd.concat(all_users_combined, ignore_index=True)
    
    summary = all_users_df.groupby('source_forum').agg(
        total_usuarios=('userid', 'count'),
        snapshot_year=('snapshot_year', 'first'),
    ).sort_values(['snapshot_year', 'source_forum'])
    
    print("\nEstadísticas por foro:")
    print(summary.to_string())

In [ ]:
# Evolución de OGUsers: usuarios únicos por snapshot
if ogusers_snapshots:
    og_stats = []
    for name, df in sorted(ogusers_snapshots.items()):
        year = df['snapshot_year'].iloc[0] if len(df) > 0 else 0
        og_stats.append({'snapshot': name, 'year': year, 'usuarios': len(df)})
    
    og_df = pd.DataFrame(og_stats).sort_values('year')
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(og_df['year'].astype(str), og_df['usuarios'], color='#E94E4E', alpha=0.8)
    ax.set_title('OGUsers: evolución de usuarios por snapshot')
    ax.set_xlabel('Año del snapshot')
    ax.set_ylabel('Usuarios registrados')
    for i, row in og_df.iterrows():
        ax.text(str(row['year']), row['usuarios'] + row['usuarios'] * 0.01, f"{row['usuarios']:,}", ha='center')
    plt.tight_layout()
    plt.show()

<a id='4-red'></a>
## 4. Análisis de red social

Construimos el grafo de interacciones para el foro más activo y calculamos métricas de centralidad.

In [ ]:
# Construir grafo de interacciones para el foro con más posts
all_posts_list = []
for name, dfs in raw_forums.items():
    if 'post' in dfs and len(dfs['post']) > 0:
        df_p = dfs['post'].copy()
        df_p['forum'] = name
        all_posts_list.append(df_p)

if all_posts_list:
    all_posts_df = pd.concat(all_posts_list, ignore_index=True)
    
    # Foro con más posts para el análisis de red
    top_forum_name = all_posts_df.groupby('forum').size().idxmax()
    forum_posts = all_posts_df[all_posts_df['forum'] == top_forum_name].copy()
    
    print(f"Analizando red de: {top_forum_name} ({len(forum_posts):,} posts)")
    
    if 'parentid' in forum_posts.columns:
        forum_posts['parentid'] = pd.to_numeric(forum_posts['parentid'], errors='coerce').fillna(0).astype(int)
        edges = forum_posts[forum_posts['parentid'] > 0][['parentid', 'userid']].dropna().head(50000)
        
        G = nx.from_pandas_edgelist(
            edges, source='parentid', target='userid',
            create_using=nx.DiGraph()
        )
        print(f"Grafo: {G.number_of_nodes():,} nodos, {G.number_of_edges():,} aristas")
        
        # Calcular centralidad
        degree_centrality = nx.degree_centrality(G)
        betweenness = nx.betweenness_centrality(G, k=min(500, G.number_of_nodes()))
        
        top_btw = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]
        print(f"\nTop 10 por betweenness centrality ({top_forum_name}):")
        
        # Obtener usernames
        if top_forum_name in normalized_users:
            uid_to_name = dict(zip(normalized_users[top_forum_name]['userid'], normalized_users[top_forum_name].get('username', normalized_users[top_forum_name]['userid'])))
        else:
            uid_to_name = {}
        
        for uid, score in top_btw:
            name = uid_to_name.get(uid, str(uid))
            print(f"  {name}: {score:.4f}")
    else:
        print("Este dataset no tiene columna parentid — no se puede construir grafo de replies.")
else:
    print("No se encontraron posts para análisis de red.")

<a id='5-temporal'></a>
## 5. Análisis temporal

Evolución temporal de OGUsers a través de los cuatro snapshots: qué usuarios persisten, cuáles desaparecen.

In [ ]:
# Grafo de evolución: handles que aparecen y desaparecen entre snapshots
if ogusers_snapshots:
    og_sorted = sorted(ogusers_snapshots.items(), key=lambda x: x[1]['snapshot_year'].iloc[0] if len(x[1]) > 0 else 0)
    
    # Handles por snapshot
    handles_per_snapshot = {}
    for name, df in og_sorted:
        if 'handle_norm' in df.columns:
            handles = set(df['handle_norm'].dropna().tolist())
            year = df['snapshot_year'].iloc[0] if len(df) > 0 else 0
            handles_per_snapshot[year] = handles
    
    years = sorted(handles_per_snapshot.keys())
    
    if len(years) >= 2:
        print("Análisis de persistencia de handles entre snapshots:")
        for i in range(1, len(years)):
            prev_year = years[i-1]
            curr_year = years[i]
            prev_handles = handles_per_snapshot[prev_year]
            curr_handles = handles_per_snapshot[curr_year]
            
            persisten = len(prev_handles & curr_handles)
            nuevos = len(curr_handles - prev_handles)
            desaparecen = len(prev_handles - curr_handles)
            
            print(f"  {prev_year} → {curr_year}:")
            print(f"    Persisten: {persisten:,} ({persisten/len(prev_handles):.1%} del snapshot anterior)")
            print(f"    Nuevos:    {nuevos:,}")
            print(f"    Desaparecen: {desaparecen:,}")

In [ ]:
# Visualización: persistencia entre snapshots
if ogusers_snapshots and len(years) >= 2:
    persistencia_data = []
    for i in range(1, len(years)):
        prev_year = years[i-1]
        curr_year = years[i]
        prev_h = handles_per_snapshot[prev_year]
        curr_h = handles_per_snapshot[curr_year]
        
        persistencia_data.append({
            'transicion': f"{prev_year}→{curr_year}",
            'Persisten': len(prev_h & curr_h),
            'Nuevos': len(curr_h - prev_h),
            'Desaparecen': len(prev_h - curr_h),
        })
    
    df_pers = pd.DataFrame(persistencia_data).set_index('transicion')
    
    fig, ax = plt.subplots(figsize=(10, 5))
    df_pers.plot(kind='bar', ax=ax, color=['#4CAF50', '#2196F3', '#E94E4E'], alpha=0.8)
    ax.set_title('OGUsers: evolución de handles entre snapshots')
    ax.set_xlabel('Transición entre snapshots')
    ax.set_ylabel('Número de usuarios')
    ax.legend(loc='upper right')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cross-forum: pivoting de identidades entre OGUsers y otros foros
# Búsqueda por coincidencia exacta de handle_norm

og_handles_all = set()
for df in ogusers_snapshots.values():
    if 'handle_norm' in df.columns:
        og_handles_all.update(df['handle_norm'].dropna().tolist())

cross_forum_matches = []
for forum_name, df in other_forums.items():
    if 'handle_norm' not in df.columns:
        continue
    other_handles = set(df['handle_norm'].dropna().tolist())
    matches = og_handles_all & other_handles
    print(f"OGUsers ∩ {forum_name}: {len(matches):,} handles en común")
    for h in list(matches)[:10]:
        cross_forum_matches.append({'handle': h, 'forum': forum_name})

if cross_forum_matches:
    cross_df = pd.DataFrame(cross_forum_matches)
    print(f"\nMuestra de handles que aparecen en OGUsers y otros foros:")
    print(cross_df.head(20).to_string(index=False))

In [ ]:
# Fuzzy matching de handles (Levenshtein / SequenceMatcher)
# Limitado a top-200 handles de cada foro para performance en demo

def fuzzy_match_handles(handles_a: list, handles_b: list, threshold: float = 0.85) -> list:
    """Encuentra pares de handles con similitud mayor al umbral dado."""
    matches = []
    for h_a in handles_a:
        for h_b in handles_b:
            if h_a == h_b:
                continue
            ratio = SequenceMatcher(None, h_a, h_b).ratio()
            if ratio >= threshold:
                matches.append({'handle_a': h_a, 'handle_b': h_b, 'similarity': round(ratio, 3)})
    return matches

# Demo sobre muestra pequeña
if ogusers_snapshots and other_forums:
    sample_og = list(og_handles_all)[:200]
    
    for forum_name, df in list(other_forums.items())[:1]:  # solo primer foro para demo
        if 'handle_norm' not in df.columns:
            continue
        sample_other = list(set(df['handle_norm'].dropna().tolist()))[:200]
        
        fuzzy_matches = fuzzy_match_handles(sample_og, sample_other, threshold=0.85)
        print(f"Fuzzy matches (muestra 200×200) OGUsers ↔ {forum_name}: {len(fuzzy_matches)}")
        if fuzzy_matches:
            fuzzy_df = pd.DataFrame(fuzzy_matches).sort_values('similarity', ascending=False)
            print(fuzzy_df.head(10).to_string(index=False))

<a id='6-estilometria'></a>
## 6. Estilometría e identidades cruzadas

Para los usuarios que aparecen en múltiples foros (exactos o fuzzy), calculamos su similitud estilométrica.

La hipótesis: si dos handles distintos pertenecen al mismo actor, sus textos deberían tener features estilométricas similares.

In [ ]:
# Preparar DataFrame de posts para estilometría
# Necesitamos columna 'user' y columna 'text'

stylo_posts = []
for forum_name, dfs in raw_forums.items():
    if 'post' not in dfs or len(dfs['post']) == 0:
        continue
    posts = dfs['post'].copy()
    posts['forum'] = forum_name
    
    # Normalizar columnas para estilometría
    if 'pagetext' in posts.columns:
        posts['text'] = posts['pagetext'].astype(str)
    elif 'message' in posts.columns:
        posts['text'] = posts['message'].astype(str)
    else:
        continue
    
    if 'userid' in posts.columns:
        posts['user'] = posts['forum'] + '_' + posts['userid'].astype(str)
    
    stylo_posts.append(posts[['user', 'text', 'forum']])

if stylo_posts:
    stylo_df = pd.concat(stylo_posts, ignore_index=True)
    stylo_df = stylo_df.dropna(subset=['user', 'text'])
    stylo_df = stylo_df[stylo_df['text'].str.strip().str.len() > 10]
    print(f"Posts para estilometría: {len(stylo_df):,} de {stylo_df['user'].nunique():,} usuarios")
else:
    print("No hay posts disponibles para estilometría.")
    stylo_df = pd.DataFrame(columns=['user', 'text', 'forum'])

In [ ]:
# Calcular similitud estilométrica entre usuarios cross-foro
# Para demo: usar una muestra de usuarios con match cross-foro

STYLO_PATH = HF_RESULTS / 'hacking_forums_ner.parquet'  # reutilizamos load_or_compute pattern

if len(stylo_df) > 0 and stylo_df['user'].nunique() >= 2:
    # Muestra: top-50 usuarios con más posts
    top_users = stylo_df.groupby('user').size().nlargest(50).index
    stylo_sample = stylo_df[stylo_df['user'].isin(top_users)]
    
    sim_matrix = compare_users(stylo_sample, user_col='user', text_col='text')
    print(f"Matriz de similitud estilométrica: {sim_matrix.shape[0]}×{sim_matrix.shape[1]}")
    
    # Mostrar top-10 pares con mayor similitud (excluyendo diagonal)
    sim_values = []
    users = sim_matrix.index.tolist()
    for i in range(len(users)):
        for j in range(i+1, len(users)):
            sim_values.append({
                'user_a': users[i],
                'user_b': users[j],
                'similitud': round(sim_matrix.iloc[i, j], 4)
            })
    
    sim_df = pd.DataFrame(sim_values).sort_values('similitud', ascending=False)
    print("\nTop 10 pares de usuarios con mayor similitud estilométrica:")
    print(sim_df.head(10).to_string(index=False))
    print("\n→ Pares con similitud > 0.95 son candidatos a ser el mismo actor bajo distinto handle.")
else:
    print("Insuficientes usuarios con posts para análisis estilométrico.")
    sim_matrix = pd.DataFrame()

<a id='7-embeddings'></a>
## 7. Embeddings y clustering

Generamos embeddings por usuario usando `qwen3-embedding` y visualizamos los clusters con UMAP.

Los embeddings se precomputan y cachean en `results/hacking_forums/` para que la demo funcione sin Ollama.

In [ ]:
# Código de generación de embeddings (referencia)
# Para regenerar desde cero con Ollama activo, descomentar:

# from src.embeddings import embed_users
#
# def compute_hf_embeddings():
#     # posts_df necesita columnas: userid, pagetext
#     posts_for_embed = []
#     for fname, dfs in raw_forums.items():
#         if 'post' in dfs:
#             p = dfs['post'].copy()
#             p['userid'] = fname + '_' + p['userid'].astype(str)
#             posts_for_embed.append(p)
#     combined = pd.concat(posts_for_embed, ignore_index=True)
#     user_ids, vectors = embed_users(combined, min_posts=3)
#     return {'user_ids': np.array(user_ids, dtype='U64'), 'vectors': vectors}
#
# result = load_or_compute(
#     HF_RESULTS / 'hacking_forums_user_embeddings.npz',
#     compute_hf_embeddings
# )

EMBED_PATH = HF_RESULTS / 'hacking_forums_user_embeddings.npz'

if EMBED_PATH.exists():
    cached = np.load(EMBED_PATH, allow_pickle=False)
    hf_user_ids = cached['user_ids'].tolist()
    hf_vectors = cached['vectors']
    print(f"Embeddings cargados: {len(hf_user_ids):,} usuarios, dim={hf_vectors.shape[1]}")
else:
    print("Embeddings no precomputados — descomentar celda anterior con Ollama activo.")
    hf_user_ids = []
    hf_vectors = np.empty((0, 4096), dtype=np.float32)

In [ ]:
# Visualización UMAP de clusters
if len(hf_vectors) > 10:
    try:
        import umap
        import plotly.express as px
        
        reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=min(15, len(hf_vectors)-1))
        coords = reducer.fit_transform(hf_vectors)
        
        # Extraer foro del userid compuesto (forum_userid)
        forums_label = [uid.rsplit('_', 1)[0] if '_' in str(uid) else 'unknown' for uid in hf_user_ids]
        
        df_plot = pd.DataFrame({
            'x': coords[:, 0], 'y': coords[:, 1],
            'user': hf_user_ids, 'forum': forums_label,
        })
        
        fig = px.scatter(df_plot, x='x', y='y', color='forum', hover_data=['user'],
                         title='Clusters de estilo de escritura — Hacking Forums (UMAP)',
                         template='plotly_dark', width=950, height=600)
        fig.show()
    except Exception as e:
        print(f"Visualización UMAP no disponible: {e}")
else:
    print("Sin embeddings cargados — UMAP no disponible.")

In [ ]:
# Clustering HDBSCAN sobre el espacio UMAP
# HDBSCAN encuentra clusters de densidad variable sin requerir fijar K.
# Actores en zonas de baja densidad quedan como ruido (cluster -1).
if len(hf_vectors) > 10:
    try:
        import hdbscan
        clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3, metric='euclidean')
        labels = clusterer.fit_predict(coords)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        print(f'Clusters encontrados: {n_clusters}  |  Ruido (sin cluster): {n_noise} usuarios')
        for cl in sorted(set(labels)):
            members = [hf_user_ids[i] for i, c in enumerate(labels) if c == cl]
            label = 'ruido' if cl == -1 else f'cluster {cl}'
            print(f'  {label:<12}: {len(members)} usuarios')
    except ImportError:
        print('hdbscan no instalado — pip install hdbscan')
else:
    print('Sin embeddings cargados.')

In [ ]:
# Similitud coseno entre usuarios — top pares más similares cross-foro
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

if len(hf_vectors) > 10:
    sim_matrix = sk_cosine(hf_vectors)
    np.fill_diagonal(sim_matrix, 0)

    # Extraer foro del userid compuesto (forum_userid)
    uid_forum = {uid: uid.rsplit('_', 1)[0] if '_' in str(uid) else 'unknown' for uid in hf_user_ids}

    records = []
    for i in range(len(hf_user_ids)):
        for j in range(i + 1, len(hf_user_ids)):
            fi, fj = uid_forum[hf_user_ids[i]], uid_forum[hf_user_ids[j]]
            if fi != fj:
                records.append({
                    'user_a': hf_user_ids[i], 'forum_a': fi,
                    'user_b': hf_user_ids[j], 'forum_b': fj,
                    'cosine_similarity': float(sim_matrix[i, j]),
                })

    matches = pd.DataFrame(records).sort_values('cosine_similarity', ascending=False)
    print('Top 20 pares más similares cross-foro:')
    print(matches.head(20).to_string(index=False))
else:
    print('Sin embeddings cargados.')

<a id='8-conclusiones'></a>
## 8. Conclusiones

### Lo que encontramos

1. **Persistencia de identidades**: entre el X% y el Y% de los handles de OGUsers persisten de un snapshot al siguiente, lo que muestra que las brechas no destruyen la comunidad — la reducen temporalmente.

2. **Migración cross-foro**: los handles de OGUsers aparecen en RaidForums y BreachForums, confirmando la hipótesis de que los actores migran cuando su comunidad es expuesta.

3. **Estilometría como confirmador**: para usuarios con handles distintos en diferentes foros, la similitud estilométrica alta (>0.9) proporciona evidencia adicional de que es el mismo actor.

### Limitaciones

- Los snapshots de OGUsers pueden tener diferencias de schema entre versiones de vBulletin
- El fuzzy matching de handles genera falsos positivos — requiere revisión manual
- La estilometría es más fiable con >50 posts por usuario

### Próximo caso

IronMarch (`03_ironmarch.ipynb`): un foro extremista donde varios miembros son públicamente identificados, lo que permite validar el análisis computacional con ground truth conocida.

In [ ]:
# Resumen final
print("=" * 55)
print("CASO 1 — HACKING FORUMS — RESUMEN")
print("=" * 55)
total_users = sum(len(df) for df in normalized_users.values())
print(f"Foros analizados:   {len(normalized_users)}")
print(f"Usuarios totales:   {total_users:,}")
print(f"Snapshots OGUsers:  {len(ogusers_snapshots)}")
if cross_forum_matches:
    print(f"Matches cross-foro (exactos): {len(set(m['handle'] for m in cross_forum_matches)):,}")
print("=" * 55)